In [ ]:
# ============================================================
# 6-HOUR AUDIT TRIAL — CELL 1
# Load and clean 6-hour cumulative smart-meter files
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import pearsonr, spearmanr
import matplotlib.pyplot as plt

OUT = Path("sixhour_outputs")
OUT.mkdir(exist_ok=True)

FIG_DIR = Path("final_figures")
FIG_DIR.mkdir(exist_ok=True)

# Put smart_6hour_1.csv ... smart_6hour_5.csv in this notebook folder
sixhour_files = sorted(Path(".").glob("smart_6hour_*.csv"))

print("6-hour files found:")
for f in sixhour_files:
    print(" -", f)

if len(sixhour_files) == 0:
    raise FileNotFoundError("No smart_6hour_*.csv files found. Copy the 6-hour files into this folder first.")

required_cols = ["household_ID", "BRANCH", "DEPOT", "E_BILLING", "EXPORT", "DATE", "TIME"]

cum_cols = [
    "TOTAL_IMPORT (kWh)",
    "TOTAL_EXPORT (kWh)",
    "TR1_TOTAL_IMPORT (kWh)",
    "TR2_TOTAL_IMPORT (kWh)",
    "TR3_TOTAL_IMPORT (kWh)",
    "TR1_TOTAL_EXPORT (kWh)",
    "TR2_TOTAL_EXPORT (kWh)",
    "TR3_TOTAL_EXPORT (kWh)"
]

parts = []

for fp in sixhour_files:
    df = pd.read_csv(fp, low_memory=False)
    df.columns = df.columns.str.strip()
    df["_source_file"] = fp.name

    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{fp.name} missing required columns: {missing}")

    available_cum = [c for c in cum_cols if c in df.columns]
    if len(available_cum) == 0:
        raise ValueError(f"{fp.name} has no expected cumulative kWh columns.")

    keep_cols = required_cols + available_cum + ["_source_file"]
    parts.append(df[keep_cols].copy())

sixh = pd.concat(parts, ignore_index=True)

sixh["household_ID"] = sixh["household_ID"].astype(str).str.strip()

# Parse datetime
dt_text = sixh["DATE"].astype(str).str.strip() + " " + sixh["TIME"].astype(str).str.strip()
sixh["reading_datetime"] = pd.to_datetime(dt_text, errors="coerce")

if sixh["reading_datetime"].isna().mean() > 0.10:
    sixh["reading_datetime"] = pd.to_datetime(dt_text, errors="coerce", dayfirst=True)

sixh = sixh.dropna(subset=["reading_datetime"]).copy()

# Convert cumulative columns to numeric
available_cum = [c for c in cum_cols if c in sixh.columns]

for c in available_cum:
    sixh[c] = (
        sixh[c]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
        .replace({"": np.nan, "nan": np.nan, "None": np.nan})
    )
    sixh[c] = pd.to_numeric(sixh[c], errors="coerce")

# Remove duplicate household timestamp records
before = len(sixh)

sixh = (
    sixh.sort_values(["household_ID", "reading_datetime", "_source_file"])
    .drop_duplicates(subset=["household_ID", "reading_datetime"], keep="last")
    .copy()
)

after = len(sixh)

print("Combined shape:", sixh.shape)
print("Unique households:", sixh["household_ID"].nunique())
print("Date range:", sixh["reading_datetime"].min(), "to", sixh["reading_datetime"].max())
print("Duplicate household-timestamp rows removed:", before - after)

sixh.to_csv(OUT / "sixhour_clean_timestamped.csv", index=False)

display(sixh.head())

In [ ]:
# ============================================================
# 6-HOUR AUDIT TRIAL — CELL 2
# Compute deltas from cumulative registers
# ============================================================


sixh = pd.read_csv(OUT / "sixhour_clean_timestamped.csv", low_memory=False)
sixh["reading_datetime"] = pd.to_datetime(sixh["reading_datetime"])
sixh["household_ID"] = sixh["household_ID"].astype(str)

sixh = sixh.sort_values(["household_ID", "reading_datetime"]).copy()

sixh["prev_datetime"] = sixh.groupby("household_ID")["reading_datetime"].shift(1)
sixh["gap_hours"] = (
    sixh["reading_datetime"] - sixh["prev_datetime"]
).dt.total_seconds() / 3600

# Register meanings are defined in the supplied dataset description:
# TR1 = Day, TR2 = Peak, TR3 = Off-peak.
# The mapping is therefore retained as documented metadata rather than inferred
# from the observed data.
rename_map = {
    "TOTAL_IMPORT (kWh)": "total_import",
    "TOTAL_EXPORT (kWh)": "total_export",
    "TR1_TOTAL_IMPORT (kWh)": "day_import",
    "TR2_TOTAL_IMPORT (kWh)": "peak_import",
    "TR3_TOTAL_IMPORT (kWh)": "offpeak_import",
    "TR1_TOTAL_EXPORT (kWh)": "day_export",
    "TR2_TOTAL_EXPORT (kWh)": "peak_export",
    "TR3_TOTAL_EXPORT (kWh)": "offpeak_export"
}

# Physics/anomaly cap:
# 24 kWh/hour is equivalent to the 15-minute pipeline cap of 6 kWh per 15 minutes.
# Because the cap is applied as a rate, a 6-hour gap allows up to 24 × 6 = 144 kWh.
MAX_KWH_PER_HOUR = 24.0
quality_rows = []

for original_col, short_name in rename_map.items():
    if original_col not in sixh.columns:
        continue

    delta_col = f"{short_name}_delta_kwh"
    rate_col = f"{short_name}_kwh_per_hour"

    sixh[delta_col] = sixh.groupby("household_ID")[original_col].diff()
    sixh[rate_col] = sixh[delta_col] / sixh["gap_hours"]

    valid = (
        sixh["gap_hours"].notna()
        & (sixh["gap_hours"] > 0)
        & sixh[delta_col].notna()
        & (sixh[delta_col] >= 0)
        & (sixh[rate_col] <= MAX_KWH_PER_HOUR)
    )

    quality_rows.append({
        "register": original_col,
        "delta_col": delta_col,
        "rows": len(sixh),
        "valid_deltas": int(valid.sum()),
        "valid_rate": float(valid.mean()),
        "negative_deltas": int((sixh[delta_col] < 0).sum()),
        "above_power_cap": int((sixh[rate_col] > MAX_KWH_PER_HOUR).sum())
    })

    sixh.loc[~valid, delta_col] = np.nan

quality = pd.DataFrame(quality_rows)

print("Gap hours summary:")
display(sixh["gap_hours"].describe(percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]))

print("Delta quality:")
display(quality)

sixh.to_csv(OUT / "sixhour_delta_cleaned.csv", index=False)
quality.to_csv(OUT / "sixhour_delta_quality_summary.csv", index=False)

In [ ]:
# ============================================================
# CELL 2B: Correct monthly allocation at calendar boundaries
# ============================================================
# Main rule:
# - Same-month physically valid deltas are retained fully.
# - Adjacent-month intervals with gaps <= 24 hours are split
#   proportionally by the hours falling in each calendar month.
# - Longer or non-adjacent cross-month gaps are excluded from
#   monthly reconstruction because their month-level distribution
#   cannot be inferred reliably.
# - Annual household totals remain based on all valid deltas.

import pandas as pd
import numpy as np
from pathlib import Path

OUT = Path("sixhour_outputs")

delta = pd.read_csv(
    OUT / "sixhour_delta_cleaned.csv",
    low_memory=False
)

delta["household_ID"] = (
    delta["household_ID"].astype(str).str.strip()
)

delta["reading_datetime"] = pd.to_datetime(
    delta["reading_datetime"],
    errors="coerce"
)

delta["prev_datetime"] = pd.to_datetime(
    delta["prev_datetime"],
    errors="coerce"
)

delta["gap_hours"] = pd.to_numeric(
    delta["gap_hours"],
    errors="coerce"
)

delta_cols = [
    c for c in delta.columns
    if c.endswith("_delta_kwh")
]

for col in delta_cols:
    delta[col] = pd.to_numeric(
        delta[col],
        errors="coerce"
    )

START = pd.Timestamp("2023-04-01")
END = pd.Timestamp("2024-04-01")

# Note:
# Intervals crossing the outer study-window boundaries are excluded unless both
# endpoints are within 2023-04-01 to 2024-04-01. This is a conservative choice
# and may slightly undercount the first/last month, but avoids allocating energy
# from outside the defined validation window.

delta["_interval_id"] = np.arange(
    len(delta),
    dtype=np.int64
)

delta["_current_period"] = (
    delta["reading_datetime"].dt.to_period("M")
)

delta["_previous_period"] = (
    delta["prev_datetime"].dt.to_period("M")
)

delta["_current_month_index"] = (
    delta["reading_datetime"].dt.year * 12
    + delta["reading_datetime"].dt.month
)

delta["_previous_month_index"] = (
    delta["prev_datetime"].dt.year * 12
    + delta["prev_datetime"].dt.month
)

valid_base = (
    delta["reading_datetime"].notna()
    & delta["prev_datetime"].notna()
    & (delta["reading_datetime"] >= START)
    & (delta["reading_datetime"] < END)
    & (delta["prev_datetime"] >= START)
    & (delta["prev_datetime"] < END)
    & delta["gap_hours"].notna()
    & (delta["gap_hours"] > 0)
    & delta["total_import_delta_kwh"].notna()
)

same_month = (
    delta["_current_period"]
    == delta["_previous_period"]
)

adjacent_month = (
    delta["_current_month_index"]
    - delta["_previous_month_index"]
    == 1
)

short_adjacent_cross_month = (
    valid_base
    & (~same_month)
    & adjacent_month
    & (delta["gap_hours"] <= 24)
)

same_rows = delta.loc[
    valid_base & same_month
].copy()

same_rows["month"] = (
    same_rows["_current_period"].astype(str)
)

same_rows["allocation_weight"] = 1.0
same_rows["allocation_type"] = "same_month"

cross = delta.loc[
    short_adjacent_cross_month
].copy()

boundary = (
    cross["reading_datetime"]
    .dt.to_period("M")
    .dt.start_time
)

cross["_hours_in_previous_month"] = (
    boundary - cross["prev_datetime"]
).dt.total_seconds() / 3600

cross["_hours_in_current_month"] = (
    cross["reading_datetime"] - boundary
).dt.total_seconds() / 3600

cross["_allocated_hours_total"] = (
    cross["_hours_in_previous_month"]
    + cross["_hours_in_current_month"]
)

cross = cross[
    cross["_allocated_hours_total"] > 0
].copy()

cross["_previous_weight"] = (
    cross["_hours_in_previous_month"]
    / cross["_allocated_hours_total"]
)

cross["_current_weight"] = (
    cross["_hours_in_current_month"]
    / cross["_allocated_hours_total"]
)

previous_rows = cross.copy()
previous_rows["month"] = (
    previous_rows["_previous_period"].astype(str)
)
previous_rows["allocation_weight"] = (
    previous_rows["_previous_weight"]
)
previous_rows["allocation_type"] = (
    "adjacent_cross_month_previous"
)

current_rows = cross.copy()
current_rows["month"] = (
    current_rows["_current_period"].astype(str)
)
current_rows["allocation_weight"] = (
    current_rows["_current_weight"]
)
current_rows["allocation_type"] = (
    "adjacent_cross_month_current"
)

for col in delta_cols:
    previous_rows[col] = (
        previous_rows[col]
        * previous_rows["allocation_weight"]
    )
    current_rows[col] = (
        current_rows[col]
        * current_rows["allocation_weight"]
    )

allocated = pd.concat(
    [same_rows, previous_rows, current_rows],
    ignore_index=True,
    sort=False
)

allocated = allocated[
    allocated["allocation_weight"] > 0
].copy()

keep_cols = [
    "_interval_id",
    "household_ID",
    "prev_datetime",
    "reading_datetime",
    "gap_hours",
    "month",
    "allocation_weight",
    "allocation_type"
] + delta_cols

allocated = allocated[keep_cols].copy()

allocated.to_csv(
    OUT / "sixhour_month_allocated_intervals.csv",
    index=False
)

cross_all = valid_base & (~same_month)
excluded_cross = cross_all & (~short_adjacent_cross_month)

same_energy = delta.loc[
    valid_base & same_month,
    "total_import_delta_kwh"
].sum()

short_cross_energy = delta.loc[
    short_adjacent_cross_month,
    "total_import_delta_kwh"
].sum()

allocated_energy = (
    allocated["total_import_delta_kwh"].sum()
)

excluded_cross_energy = delta.loc[
    excluded_cross,
    "total_import_delta_kwh"
].sum()

allocation_audit = pd.DataFrame([{
    "same_month_interval_rows": int(
        (valid_base & same_month).sum()
    ),
    "short_adjacent_cross_month_intervals": int(
        short_adjacent_cross_month.sum()
    ),
    "allocated_segment_rows": int(len(allocated)),
    "same_month_import_kwh": float(same_energy),
    "short_cross_month_import_kwh": float(
        short_cross_energy
    ),
    "allocated_import_kwh": float(allocated_energy),
    "allocation_conservation_difference_kwh": float(
        allocated_energy
        - same_energy
        - short_cross_energy
    ),
    "excluded_cross_month_intervals": int(
        excluded_cross.sum()
    ),
    "excluded_cross_month_import_kwh": float(
        excluded_cross_energy
    )
}])

allocation_audit.to_csv(
    OUT / "STEP42_cross_month_allocation_audit.csv",
    index=False
)

print("Corrected monthly allocation audit:")
display(allocation_audit)

if not np.isclose(
    allocation_audit.loc[
        0,
        "allocation_conservation_difference_kwh"
    ],
    0.0,
    atol=1e-6
):
    raise AssertionError(
        "Allocated energy was not conserved."
    )

print(
    "Saved:",
    OUT / "sixhour_month_allocated_intervals.csv"
)


In [ ]:
# ============================================================
# 6-HOUR AUDIT TRIAL — CELL 3 CORRECTED
# Validate month-allocated 6-hour import against monthly billing
# Uses raw monthly_consumption.csv and filters Apr 2023–Mar 2024
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import pearsonr, spearmanr
import matplotlib.pyplot as plt

OUT = Path("sixhour_outputs")
FIG_DIR = Path("final_figures")
FIG_DIR.mkdir(exist_ok=True)

monthly_path = Path("monthly_consumption.csv")

if not monthly_path.exists():
    raise FileNotFoundError(
        "Copy monthly_consumption.csv into this notebook folder first."
    )

monthly = pd.read_csv(monthly_path)
monthly.columns = monthly.columns.str.strip()

monthly["household_ID"] = (
    monthly["household_ID"].astype(str).str.strip()
)

monthly["month"] = (
    pd.to_datetime(
        monthly["month"],
        errors="coerce"
    )
    .dt.to_period("M")
    .astype(str)
)

monthly["consumption"] = (
    monthly["consumption"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.strip()
)

monthly["consumption"] = pd.to_numeric(
    monthly["consumption"],
    errors="coerce"
)

monthly = monthly[
    monthly["month"].between(
        "2023-04",
        "2024-03"
    )
].copy()

allocated = pd.read_csv(
    OUT / "sixhour_month_allocated_intervals.csv",
    low_memory=False
)

allocated["household_ID"] = (
    allocated["household_ID"].astype(str).str.strip()
)

allocated["month"] = allocated["month"].astype(str)

allocated["total_import_delta_kwh"] = pd.to_numeric(
    allocated["total_import_delta_kwh"],
    errors="coerce"
)

if "total_export_delta_kwh" in allocated.columns:
    allocated["total_export_delta_kwh"] = pd.to_numeric(
        allocated["total_export_delta_kwh"],
        errors="coerce"
    )
else:
    allocated["total_export_delta_kwh"] = np.nan

allocated["gap_hours"] = pd.to_numeric(
    allocated["gap_hours"],
    errors="coerce"
)

allocated["export_interval_id"] = np.where(
    allocated["total_export_delta_kwh"].notna(),
    allocated["_interval_id"],
    np.nan
)

sixh_monthly = (
    allocated
    .groupby(
        ["household_ID", "month"],
        as_index=False
    )
    .agg(
        sixh_import_kwh=(
            "total_import_delta_kwh",
            "sum"
        ),
        sixh_export_kwh=(
            "total_export_delta_kwh",
            lambda s: s.sum(min_count=1)
        ),
        sixh_export_valid_delta_count=(
            "export_interval_id",
            "nunique"
        ),
        sixh_readings=(
            "_interval_id",
            "nunique"
        ),
        median_gap_hours=(
            "gap_hours",
            "median"
        )
    )
)

sixh_monthly.to_csv(
    OUT / "sixhour_monthly_reconstructed_corrected.csv",
    index=False
)

val = monthly.merge(
    sixh_monthly,
    on=["household_ID", "month"],
    how="inner",
    validate="one_to_one"
)

val = val.dropna(
    subset=[
        "consumption",
        "sixh_import_kwh"
    ]
).copy()

val["signed_error_kwh"] = (
    val["sixh_import_kwh"]
    - val["consumption"]
)

val["abs_error_kwh"] = (
    val["signed_error_kwh"].abs()
)

val["ape"] = np.where(
    val["consumption"] > 0,
    val["abs_error_kwh"]
    / val["consumption"],
    np.nan
)

if (
    len(val) >= 10
    and val["consumption"].nunique() > 1
    and val["sixh_import_kwh"].nunique() > 1
):
    pearson_r, pearson_p = pearsonr(
        val["consumption"],
        val["sixh_import_kwh"]
    )
    spearman_r, spearman_p = spearmanr(
        val["consumption"],
        val["sixh_import_kwh"]
    )
else:
    pearson_r = np.nan
    pearson_p = np.nan
    spearman_r = np.nan
    spearman_p = np.nan

summary = pd.DataFrame([{
    "allocation_method": (
        "same-month full + adjacent cross-month <=24h proportional split"
    ),
    "matched_household_months": len(val),
    "matched_households": (
        val["household_ID"].nunique()
    ),
    "pearson_r": pearson_r,
    "pearson_p": pearson_p,
    "spearman_r": spearman_r,
    "spearman_p": spearman_p,
    "median_abs_error_kwh": (
        val["abs_error_kwh"].median()
    ),
    "median_ape": val["ape"].median(),
    "mean_ape": val["ape"].mean(),
    "mean_signed_error_kwh": (
        val["signed_error_kwh"].mean()
    )
}])

display(summary)

val.to_csv(
    OUT / "sixhour_monthly_validation_rows.csv",
    index=False
)

summary.to_csv(
    OUT / "sixhour_monthly_validation_summary.csv",
    index=False
)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(
    val["consumption"],
    val["sixh_import_kwh"],
    alpha=0.35,
    s=14
)

if len(val) > 0:
    max_val = np.nanpercentile(
        pd.concat([
            val["consumption"],
            val["sixh_import_kwh"]
        ]),
        99
    )
    ax.plot(
        [0, max_val],
        [0, max_val],
        linestyle="--",
        linewidth=1
    )
    ax.set_xlim(0, max_val)
    ax.set_ylim(0, max_val)

ax.set_xlabel("Monthly billing consumption (kWh)")
ax.set_ylabel("Corrected 6-hour reconstructed import (kWh)")
ax.set_title("Corrected 6-hour import vs monthly billing")

plt.tight_layout()
plt.savefig(
    FIG_DIR / "Figure_30_6hour_vs_monthly_validation.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


In [ ]:
# Audit date ranges by 6-hour source file
file_ranges = (
    sixh
    .groupby("_source_file")
    .agg(
        rows=("household_ID", "size"),
        households=("household_ID", "nunique"),
        min_datetime=("reading_datetime", "min"),
        max_datetime=("reading_datetime", "max")
    )
    .reset_index()
)

display(file_ranges)

# Check households appearing in multiple files
hh_file_counts = (
    sixh.groupby("household_ID")["_source_file"]
    .nunique()
    .reset_index(name="n_source_files")
)

display(hh_file_counts["n_source_files"].value_counts().sort_index())

In [ ]:
# ============================================================
# 6-HOUR SUPPORT ANALYSIS — CELL 5
# Build safe monthly and household-level 6-hour features
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import kruskal
import matplotlib.pyplot as plt

OUT = Path("sixhour_outputs")
FIG_DIR = Path("final_figures")
FIG_DIR.mkdir(exist_ok=True)

sixh = pd.read_csv(OUT / "sixhour_delta_cleaned.csv", low_memory=False)

sixh["household_ID"] = sixh["household_ID"].astype(str).str.strip()
sixh["reading_datetime"] = pd.to_datetime(sixh["reading_datetime"])
sixh["prev_datetime"] = pd.to_datetime(sixh["prev_datetime"])

START = pd.Timestamp("2023-04-01")
END = pd.Timestamp("2024-04-01")

sixh["month"] = sixh["reading_datetime"].dt.to_period("M").astype(str)
sixh["prev_month"] = sixh["prev_datetime"].dt.to_period("M").astype(str)

# Use only deltas inside the final study window.
valid_period = (
    (sixh["reading_datetime"] >= START)
    & (sixh["reading_datetime"] < END)
    & (sixh["prev_datetime"] >= START)
)

# For monthly reconstruction, use the corrected calendar-boundary allocation
# created in Cell 2B. This retains same-month deltas and proportionally splits
# short adjacent-month intervals (<=24 hours).
corrected_monthly_path = (
    OUT / "sixhour_monthly_reconstructed_corrected.csv"
)

if not corrected_monthly_path.exists():
    raise FileNotFoundError(
        "Run Cell 2B and corrected Cell 3 before this cell."
    )

sixh_monthly = pd.read_csv(
    corrected_monthly_path
)

sixh_monthly["household_ID"] = (
    sixh_monthly["household_ID"]
    .astype(str)
    .str.strip()
)

sixh_monthly["month"] = (
    sixh_monthly["month"].astype(str)
)

# Build annual household features using all valid deltas inside the study window.
annual_cols = {
    "sixh_total_import_kwh": ("total_import_delta_kwh", "sum"),
    "sixh_total_export_kwh": ("total_export_delta_kwh", "sum"),
    "sixh_day_import_kwh": ("day_import_delta_kwh", "sum"),
    "sixh_peak_import_kwh": ("peak_import_delta_kwh", "sum"),
    "sixh_offpeak_import_kwh": ("offpeak_import_delta_kwh", "sum"),
    "sixh_day_export_kwh": ("day_export_delta_kwh", "sum"),
    "sixh_peak_export_kwh": ("peak_export_delta_kwh", "sum"),
    "sixh_offpeak_export_kwh": ("offpeak_export_delta_kwh", "sum"),
    "sixh_reading_count": ("reading_datetime", "count"),
    "sixh_median_gap_hours": ("gap_hours", "median")
}

# Keep only columns that exist.
annual_cols = {
    new: old for new, old in annual_cols.items()
    if old[0] in sixh.columns
}

sixh_household = (
    sixh.loc[valid_period]
    .groupby("household_ID")
    .agg(**annual_cols)
    .reset_index()
)

# Time-of-use import shares
# Use fillna(0) before summing so partial tariff-register coverage does not
# silently create NaN denominators for otherwise valid households.
def safe_numeric_series(df, col):
    if col in df.columns:
        return pd.to_numeric(df[col], errors="coerce").fillna(0)
    return pd.Series(0.0, index=df.index)

day_import = safe_numeric_series(sixh_household, "sixh_day_import_kwh")
peak_import = safe_numeric_series(sixh_household, "sixh_peak_import_kwh")
offpeak_import = safe_numeric_series(sixh_household, "sixh_offpeak_import_kwh")

import_tou_total = day_import + peak_import + offpeak_import

sixh_household["sixh_day_import_share"] = np.where(
    import_tou_total > 0,
    day_import / import_tou_total,
    np.nan
)

sixh_household["sixh_peak_import_share"] = np.where(
    import_tou_total > 0,
    peak_import / import_tou_total,
    np.nan
)

sixh_household["sixh_offpeak_import_share"] = np.where(
    import_tou_total > 0,
    offpeak_import / import_tou_total,
    np.nan
)

total_import = safe_numeric_series(sixh_household, "sixh_total_import_kwh")
total_export = safe_numeric_series(sixh_household, "sixh_total_export_kwh")

sixh_household["sixh_export_import_ratio"] = np.where(
    total_import > 0,
    total_export / total_import,
    np.nan
)

# Latest metadata from 6-hour period
meta = (
    sixh.loc[valid_period]
    .sort_values(["household_ID", "reading_datetime"])
    .groupby("household_ID", as_index=False)
    .tail(1)[["household_ID", "BRANCH", "DEPOT", "E_BILLING", "EXPORT"]]
)

sixh_household = sixh_household.merge(meta, on="household_ID", how="left")

sixh_monthly.to_csv(OUT / "STEP42_6hour_monthly_features.csv", index=False)
sixh_household.to_csv(OUT / "STEP42_6hour_household_features.csv", index=False)

print("6-hour monthly features:", sixh_monthly.shape)
print("6-hour household features:", sixh_household.shape)

display(sixh_monthly.head())
display(sixh_household.head())


In [ ]:
# ============================================================
# 6-HOUR SUPPORT ANALYSIS — CELL 6 FINAL CORRECTED
# Add Low / Medium / High groups using existing Analysis 2 profile
# Preserve unknown survey/metadata solar-export states
# ============================================================

profile_path = Path("ANALYSIS2_consumption_group_profile_dataset.csv")

sixh_monthly = pd.read_csv(OUT / "STEP42_6hour_monthly_features.csv")
sixh_household = pd.read_csv(OUT / "STEP42_6hour_household_features.csv")

sixh_monthly["household_ID"] = sixh_monthly["household_ID"].astype(str).str.strip()
sixh_household["household_ID"] = sixh_household["household_ID"].astype(str).str.strip()

if not profile_path.exists():
    raise FileNotFoundError(
        "ANALYSIS2_consumption_group_profile_dataset.csv is required for consistent final groups."
    )

profile = pd.read_csv(profile_path)
profile.columns = profile.columns.str.strip()
profile["household_ID"] = profile["household_ID"].astype(str).str.strip()

print("Profile columns:")
print(profile.columns.tolist())

# Use the existing final consumption-group column.
if "consumption_group" in profile.columns:
    group_col = "consumption_group"
elif "consumption_tertile" in profile.columns:
    group_col = "consumption_tertile"
else:
    raise ValueError(
        "Profile file must contain consumption_group or consumption_tertile."
    )

required_profile_cols = [
    "household_ID",
    group_col,
    "mean_monthly_kwh",
    "export_ever_true",
    "w1_solar_generation",
]
missing_profile_cols = [c for c in required_profile_cols if c not in profile.columns]
if missing_profile_cols:
    raise ValueError(
        f"Profile file is missing required columns: {missing_profile_cols}"
    )

group_info = profile[required_profile_cols].copy()
group_info = group_info.rename(columns={group_col: "consumption_group"})

# Clean possible group labels.
group_info["consumption_group"] = (
    group_info["consumption_group"].astype(str).str.strip()
)

numeric_map = {
    "0": "Low",
    "1": "Medium",
    "2": "High",
    "low": "Low",
    "medium": "Medium",
    "high": "High",
    "Low": "Low",
    "Medium": "Medium",
    "High": "High",
}

group_info["consumption_group"] = group_info["consumption_group"].map(
    lambda x: numeric_map.get(x, x)
)

unexpected_groups = sorted(
    set(group_info["consumption_group"].dropna().unique())
    - {"Low", "Medium", "High"}
)
if unexpected_groups:
    raise ValueError(
        f"Unexpected consumption-group labels found: {unexpected_groups}"
    )

print("Consumption group labels in profile:")
display(group_info["consumption_group"].value_counts())

# Audit survey/metadata missingness before constructing joint groups.
for col in ["w1_solar_generation", "export_ever_true"]:
    group_info[col] = pd.to_numeric(group_info[col], errors="coerce")

missingness_audit = pd.DataFrame({
    "variable": ["w1_solar_generation", "export_ever_true"],
    "missing_values": [
        int(group_info["w1_solar_generation"].isna().sum()),
        int(group_info["export_ever_true"].isna().sum()),
    ],
    "available_values": [
        int(group_info["w1_solar_generation"].notna().sum()),
        int(group_info["export_ever_true"].notna().sum()),
    ],
})
missingness_audit["missing_percent"] = (
    100 * missingness_audit["missing_values"] / len(group_info)
)

print("Solar/export source-variable missingness audit:")
display(missingness_audit)

missingness_audit.to_csv(
    OUT / "STEP42_solar_export_source_missingness_audit.csv",
    index=False,
)

# Merge groups.
sixh_monthly_grouped = sixh_monthly.merge(
    group_info,
    on="household_ID",
    how="inner",
    validate="many_to_one",
)
sixh_household_grouped = sixh_household.merge(
    group_info,
    on="household_ID",
    how="inner",
    validate="one_to_one",
)

# Preserve missing solar/export states as Unknown rather than treating them as No.
solar = pd.to_numeric(
    sixh_household_grouped["w1_solar_generation"],
    errors="coerce",
)
export = pd.to_numeric(
    sixh_household_grouped["export_ever_true"],
    errors="coerce",
)

sixh_household_grouped["solar_export_group"] = np.select(
    [
        solar.eq(0) & export.eq(0),
        solar.eq(1) & export.eq(0),
        solar.eq(0) & export.eq(1),
        solar.eq(1) & export.eq(1),
    ],
    [
        "Non-solar / Non-export",
        "Solar only",
        "Export only",
        "Solar + Export",
    ],
    default="Unknown / missing",
)

sixh_monthly_grouped.to_csv(
    OUT / "STEP42_6hour_monthly_grouped.csv",
    index=False,
)
sixh_household_grouped.to_csv(
    OUT / "STEP42_6hour_household_grouped.csv",
    index=False,
)

print("Grouped monthly rows:", sixh_monthly_grouped.shape)
print("Grouped households:", sixh_household_grouped.shape)
print("Consumption group counts after 6-hour merge:")
display(sixh_household_grouped["consumption_group"].value_counts())
print("Solar/export group counts after preserving unknown states:")
display(sixh_household_grouped["solar_export_group"].value_counts(dropna=False))


In [ ]:
# ============================================================
# 6-HOUR SUPPORT ANALYSIS — CELL 7
# Monthly reconstructed import trend by consumption group
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

OUT = Path("sixhour_outputs")
FIG_DIR = Path("final_figures")
FIG_DIR.mkdir(exist_ok=True)

sixh_monthly_grouped = pd.read_csv(OUT / "STEP42_6hour_monthly_grouped.csv")

# Ensure clean labels and ordering
sixh_monthly_grouped["month"] = sixh_monthly_grouped["month"].astype(str)
sixh_monthly_grouped["consumption_group"] = (
    sixh_monthly_grouped["consumption_group"].astype(str).str.strip()
)

group_order = ["Low", "Medium", "High"]

trend = (
    sixh_monthly_grouped
    .groupby(["month", "consumption_group"], as_index=False)
    .agg(
        mean_sixh_import_kwh=("sixh_import_kwh", "mean"),
        median_sixh_import_kwh=("sixh_import_kwh", "median"),
        households=("household_ID", "nunique")
    )
)

trend["consumption_group"] = pd.Categorical(
    trend["consumption_group"],
    categories=group_order,
    ordered=True
)

trend = trend.sort_values(["month", "consumption_group"])

trend.to_csv(
    OUT / "STEP42_6hour_monthly_trend_by_consumption_group.csv",
    index=False
)

print("Monthly trend rows:", trend.shape)
display(trend.head(12))

# Plot mean monthly reconstructed import trend
fig, ax = plt.subplots(figsize=(8.2, 4.8))

for group in group_order:
    tmp = trend[trend["consumption_group"] == group].sort_values("month")

    ax.plot(
        tmp["month"],
        tmp["mean_sixh_import_kwh"],
        marker="o",
        linewidth=1.8,
        label=group
    )

ax.set_xlabel("Month")
ax.set_ylabel("Mean reconstructed 6-hour import (kWh)")
ax.set_title("6-hour reconstructed monthly import trend by consumption group")
ax.tick_params(axis="x", rotation=45)
ax.legend(title="Consumption group", frameon=False)

plt.tight_layout()

fig_path = FIG_DIR / "Figure_31_6hour_monthly_import_trend_by_group.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", fig_path)
print("Saved:", OUT / "STEP42_6hour_monthly_trend_by_consumption_group.csv")

In [ ]:
# ============================================================
# 6-HOUR SUPPORT ANALYSIS — CELL 8 FINAL CORRECTED
# Day / Peak / Off-peak import shares by consumption group
# TR1/TR2/TR3 meanings follow the supplied dataset description
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests
from pathlib import Path

OUT = Path("sixhour_outputs")
FIG_DIR = Path("final_figures")
FIG_DIR.mkdir(exist_ok=True)

sixh_household_grouped = pd.read_csv(
    OUT / "STEP42_6hour_household_grouped.csv"
)

sixh_household_grouped["consumption_group"] = (
    sixh_household_grouped["consumption_group"].astype(str).str.strip()
)

group_order = ["Low", "Medium", "High"]

tou_cols = [
    "sixh_day_import_share",
    "sixh_peak_import_share",
    "sixh_offpeak_import_share",
]

# Keep only valid share rows.
for col in tou_cols:
    sixh_household_grouped[col] = pd.to_numeric(
        sixh_household_grouped[col],
        errors="coerce",
    )

valid_tou = sixh_household_grouped.dropna(
    subset=["consumption_group"] + tou_cols
).copy()
valid_tou = valid_tou[
    valid_tou["consumption_group"].isin(group_order)
].copy()

# Summary table.
tou_summary = (
    valid_tou
    .groupby("consumption_group", observed=True)[tou_cols]
    .agg(["count", "mean", "median"])
)

tou_summary.columns = ["_".join(col).strip() for col in tou_summary.columns]
tou_summary = tou_summary.reset_index()

tou_summary["consumption_group"] = pd.Categorical(
    tou_summary["consumption_group"],
    categories=group_order,
    ordered=True,
)

tou_summary = tou_summary.sort_values("consumption_group")

tou_summary.to_csv(
    OUT / "STEP42_6hour_tou_share_summary_by_consumption_group.csv",
    index=False,
)

print("Time-of-use share summary:")
display(tou_summary)

# Kruskal-Wallis tests across the three billing-defined consumption groups.
# The three related tests are Holm-adjusted. Epsilon-squared is reported as
# an omnibus non-parametric effect-size estimate.
test_rows = []

for col in tou_cols:
    tmp = valid_tou[["consumption_group", col]].dropna()

    grouped_values = [
        group[col].to_numpy()
        for group_name, group in tmp.groupby(
            "consumption_group",
            observed=True,
            sort=False,
        )
        if group_name in group_order and len(group) >= 5
    ]

    k_groups = len(grouped_values)
    n_obs = len(tmp)

    if k_groups >= 2:
        stat, p_value = kruskal(*grouped_values)
        epsilon_squared = max(
            0.0,
            (stat - k_groups + 1) / (n_obs - k_groups)
        ) if n_obs > k_groups else np.nan
    else:
        stat, p_value, epsilon_squared = np.nan, np.nan, np.nan

    test_rows.append({
        "feature": col,
        "n": n_obs,
        "groups_tested": k_groups,
        "kruskal_H": stat,
        "p_value": p_value,
        "epsilon_squared": epsilon_squared,
    })

tou_tests = pd.DataFrame(test_rows)

valid_p = tou_tests["p_value"].notna()
tou_tests["holm_p_value"] = np.nan
if valid_p.any():
    tou_tests.loc[valid_p, "holm_p_value"] = multipletests(
        tou_tests.loc[valid_p, "p_value"],
        method="holm",
    )[1]

tou_tests["significant_after_holm"] = (
    tou_tests["holm_p_value"] < 0.05
)

tou_tests.to_csv(
    OUT / "STEP42_6hour_tou_share_tests.csv",
    index=False,
)

print("Time-of-use share Kruskal-Wallis tests with Holm correction:")
display(tou_tests)

# Plot mean Day / Peak / Off-peak shares.
plot_df = (
    valid_tou
    .groupby("consumption_group", observed=True)[tou_cols]
    .mean()
    .reindex(group_order)
)

fig, ax = plt.subplots(figsize=(7.5, 4.8))
plot_df.plot(kind="bar", ax=ax)

ax.set_xlabel("Consumption group")
ax.set_ylabel("Mean import share")
ax.set_title("Six-hour tariff-register import shares by consumption group")
ax.legend(["Day", "Peak", "Off-peak"], frameon=False)
ax.tick_params(axis="x", rotation=0)

plt.tight_layout()

fig_path = FIG_DIR / "Figure_32_6hour_tou_import_shares_by_group.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", fig_path)
print("Saved:", OUT / "STEP42_6hour_tou_share_summary_by_consumption_group.csv")
print("Saved:", OUT / "STEP42_6hour_tou_share_tests.csv")


In [ ]:
# ============================================================
# 6-HOUR SUPPORT ANALYSIS — CELL 9
# Export/import ratio by solar/export group
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

OUT = Path("sixhour_outputs")
FIG_DIR = Path("final_figures")
FIG_DIR.mkdir(exist_ok=True)

sixh_household_grouped = pd.read_csv(OUT / "STEP42_6hour_household_grouped.csv")

sixh_household_grouped["solar_export_group"] = (
    sixh_household_grouped["solar_export_group"].astype(str).str.strip()
)

# Ensure numeric
num_cols = [
    "sixh_export_import_ratio",
    "sixh_total_export_kwh",
    "sixh_total_import_kwh"
]

for c in num_cols:
    sixh_household_grouped[c] = pd.to_numeric(
        sixh_household_grouped[c],
        errors="coerce"
    )

# Summarise export intensity
export_summary = (
    sixh_household_grouped
    .groupby("solar_export_group", as_index=False)
    .agg(
        households=("household_ID", "nunique"),
        median_export_import_ratio=("sixh_export_import_ratio", "median"),
        mean_export_import_ratio=("sixh_export_import_ratio", "mean"),
        median_export_kwh=("sixh_total_export_kwh", "median"),
        mean_export_kwh=("sixh_total_export_kwh", "mean"),
        median_import_kwh=("sixh_total_import_kwh", "median"),
        mean_import_kwh=("sixh_total_import_kwh", "mean")
    )
)

preferred_order = [
    "Non-solar / Non-export",
    "Solar only",
    "Export only",
    "Solar + Export",
    "Unknown / missing",
]

# Preserve unknown group labels rather than converting them
# into confirmed non-solar/non-export households.
sixh_household_grouped["solar_export_group"] = (
    sixh_household_grouped["solar_export_group"]
    .replace("nan", "Unknown / missing")
    .fillna("Unknown / missing")
)

# Recreate the summary after cleaning the group label.
export_summary = (
    sixh_household_grouped
    .groupby(
        "solar_export_group",
        as_index=False,
        observed=True,
        dropna=False,
    )
    .agg(
        households=("household_ID", "nunique"),
        median_export_import_ratio=(
            "sixh_export_import_ratio",
            "median",
        ),
        mean_export_import_ratio=(
            "sixh_export_import_ratio",
            "mean",
        ),
        median_export_kwh=(
            "sixh_total_export_kwh",
            "median",
        ),
        mean_export_kwh=(
            "sixh_total_export_kwh",
            "mean",
        ),
        median_import_kwh=(
            "sixh_total_import_kwh",
            "median",
        ),
        mean_import_kwh=(
            "sixh_total_import_kwh",
            "mean",
        ),
    )
)

export_summary["solar_export_group"] = pd.Categorical(
    export_summary["solar_export_group"],
    categories=preferred_order,
    ordered=True,
)

export_summary = (
    export_summary
    .sort_values("solar_export_group")
    .reset_index(drop=True)
)

numeric_summary_cols = [c for c in export_summary.columns if c != "solar_export_group"]
export_summary[numeric_summary_cols] = export_summary[numeric_summary_cols].fillna(0)
export_summary["households"] = export_summary["households"].astype(int)

export_summary["solar_export_group"] = pd.Categorical(
    export_summary["solar_export_group"],
    categories=preferred_order,
    ordered=True
)

export_summary = export_summary.sort_values("solar_export_group")

export_summary.to_csv(
    OUT / "STEP42_6hour_export_ratio_by_solar_export_group.csv",
    index=False
)

print("Export/import ratio summary by solar/export group:")
display(export_summary)

# Plot median export/import ratio
fig, ax = plt.subplots(figsize=(7.5, 4.8))

ax.bar(
    export_summary["solar_export_group"].astype(str),
    export_summary["median_export_import_ratio"]
)

ax.set_xlabel("Solar/export group")
ax.set_ylabel("Median export/import ratio")
ax.set_title("6-hour export intensity by solar/export group")
ax.tick_params(axis="x", rotation=25)

plt.tight_layout()

fig_path = FIG_DIR / "Figure_33_6hour_export_ratio_by_solar_export_group.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", fig_path)
print("Saved:", OUT / "STEP42_6hour_export_ratio_by_solar_export_group.csv")

In [ ]:
# ============================================================
# 6-HOUR SUPPORT ANALYSIS — CELL 9B FINAL CORRECTED
# Audit solar/export consistency across survey, metadata and 6-hour registers
# Preserve unknown source states rather than coding missing as No
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

OUT = Path("sixhour_outputs")

df = pd.read_csv(OUT / "STEP42_6hour_household_grouped.csv")

# Ensure numeric source and register fields.
for col in [
    "w1_solar_generation",
    "export_ever_true",
    "sixh_total_export_kwh",
    "sixh_total_import_kwh",
    "sixh_export_import_ratio",
]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Explicit source-status labels retain unknown/missing values.
df["survey_solar_status"] = np.select(
    [
        df["w1_solar_generation"].eq(1),
        df["w1_solar_generation"].eq(0),
    ],
    ["Yes", "No"],
    default="Unknown / missing",
)

df["metadata_export_status"] = np.select(
    [
        df["export_ever_true"].eq(1),
        df["export_ever_true"].eq(0),
    ],
    ["Yes", "No"],
    default="Unknown / missing",
)

# Numeric known-only fields are retained for mismatch logic.
df["survey_solar"] = df["w1_solar_generation"]
df["metadata_export"] = df["export_ever_true"]

# Register-based export flags using multiple thresholds. Missing register totals
# remain unknown instead of being silently converted to zero.
register_observed = df["sixh_total_export_kwh"].notna()
for threshold, suffix in [
    (0, "0"),
    (1, "1kwh"),
    (10, "10kwh"),
    (100, "100kwh"),
]:
    flag_col = f"sixh_export_gt_{suffix}"
    df[flag_col] = pd.Series(pd.NA, index=df.index, dtype="boolean")
    df.loc[register_observed, flag_col] = (
        df.loc[register_observed, "sixh_total_export_kwh"] > threshold
    )

# Audit source-variable missingness.
source_missingness = pd.DataFrame({
    "variable": [
        "w1_solar_generation",
        "export_ever_true",
        "sixh_total_export_kwh",
    ],
    "missing_values": [
        int(df["w1_solar_generation"].isna().sum()),
        int(df["export_ever_true"].isna().sum()),
        int(df["sixh_total_export_kwh"].isna().sum()),
    ],
    "available_values": [
        int(df["w1_solar_generation"].notna().sum()),
        int(df["export_ever_true"].notna().sum()),
        int(df["sixh_total_export_kwh"].notna().sum()),
    ],
})
source_missingness["missing_percent"] = (
    100 * source_missingness["missing_values"] / len(df)
)

print("Solar/export consistency source missingness:")
display(source_missingness)
source_missingness.to_csv(
    OUT / "STEP42_6hour_solar_export_consistency_missingness.csv",
    index=False,
)

# 1. Cross-tab: survey solar vs metadata export.
print("Survey-recorded solar vs metadata EXPORT:")
display(pd.crosstab(
    df["survey_solar_status"],
    df["metadata_export_status"],
    margins=True,
    dropna=False,
))

# 2. Cross-tab: metadata export vs 6-hour register export.
register_gt1_status = df["sixh_export_gt_1kwh"].map(
    {True: "Yes", False: "No"}
).fillna("Unknown / missing")

print("Metadata EXPORT vs six-hour export register > 1 kWh:")
display(pd.crosstab(
    df["metadata_export_status"],
    register_gt1_status,
    margins=True,
    dropna=False,
))

# 3. Cross-tab: survey solar vs 6-hour register export.
print("Survey-recorded solar vs six-hour export register > 1 kWh:")
display(pd.crosstab(
    df["survey_solar_status"],
    register_gt1_status,
    margins=True,
    dropna=False,
))

# 4. Export threshold summary by solar/export group.

preferred_order = [
    "Non-solar / Non-export",
    "Solar only",
    "Export only",
    "Solar + Export",
    "Unknown / missing",
]

# Ensure every record has one explicit group label.
df["solar_export_group"] = (
    df["solar_export_group"]
    .fillna("Unknown / missing")
    .astype(str)
)

# Summarise only groups actually observed in the data.
threshold_summary = (
    df.groupby(
        "solar_export_group",
        as_index=False,
        observed=True,
        dropna=False,
    )
    .agg(
        households=("household_ID", "nunique"),
        observed_export_registers=(
            "sixh_total_export_kwh",
            "count",
        ),
        export_gt_0=(
            "sixh_export_gt_0",
            lambda s: int(s.fillna(False).sum()),
        ),
        export_gt_1kwh=(
            "sixh_export_gt_1kwh",
            lambda s: int(s.fillna(False).sum()),
        ),
        export_gt_10kwh=(
            "sixh_export_gt_10kwh",
            lambda s: int(s.fillna(False).sum()),
        ),
        export_gt_100kwh=(
            "sixh_export_gt_100kwh",
            lambda s: int(s.fillna(False).sum()),
        ),
        median_export_kwh=(
            "sixh_total_export_kwh",
            "median",
        ),
        mean_export_kwh=(
            "sixh_total_export_kwh",
            "mean",
        ),
        median_export_import_ratio=(
            "sixh_export_import_ratio",
            "median",
        ),
    )
)

# Percentages use only households with an observed six-hour
# export register as the denominator.
for col in [
    "export_gt_0",
    "export_gt_1kwh",
    "export_gt_10kwh",
    "export_gt_100kwh",
]:
    threshold_summary[f"{col}_pct"] = np.where(
        threshold_summary["observed_export_registers"] > 0,
        (
            100
            * threshold_summary[col]
            / threshold_summary["observed_export_registers"]
        ),
        np.nan,
    )

# Apply the intended display order without creating artificial rows.
threshold_summary["solar_export_group"] = pd.Categorical(
    threshold_summary["solar_export_group"],
    categories=preferred_order,
    ordered=True,
)

threshold_summary = (
    threshold_summary
    .sort_values("solar_export_group")
    .reset_index(drop=True)
)

print(
    "Six-hour export threshold summary "
    "by solar/export group:"
)
display(threshold_summary)

threshold_summary.to_csv(
    OUT
    / "STEP42_6hour_solar_export_consistency_"
      "threshold_summary.csv",
    index=False,
)

# 5. List mismatch households only where metadata and register status are known.
known_metadata = df["metadata_export"].isin([0, 1])
known_register = df["sixh_export_gt_1kwh"].notna()

mismatch_mask = (
    known_metadata
    & known_register
    & (
        (
            df["metadata_export"].eq(0)
            & df["sixh_export_gt_10kwh"].fillna(False)
        )
        |
        (
            df["metadata_export"].eq(1)
            & ~df["sixh_export_gt_1kwh"].fillna(False)
        )
    )
)

mismatch = df.loc[mismatch_mask].copy()

mismatch_cols = [
    "household_ID",
    "solar_export_group",
    "survey_solar_status",
    "metadata_export_status",
    "sixh_total_export_kwh",
    "sixh_total_import_kwh",
    "sixh_export_import_ratio",
    "DEPOT",
    "BRANCH",
]
mismatch_cols = [c for c in mismatch_cols if c in mismatch.columns]

print("Potential known-status mismatch households:")
print("Count:", len(mismatch))
display(
    mismatch[mismatch_cols]
    .sort_values("sixh_total_export_kwh", ascending=False)
    .head(30)
)

mismatch[mismatch_cols].to_csv(
    OUT / "STEP42_6hour_solar_export_mismatch_households.csv",
    index=False,
)

print("Saved:")
print(OUT / "STEP42_6hour_solar_export_consistency_threshold_summary.csv")
print(OUT / "STEP42_6hour_solar_export_mismatch_households.csv")
print(OUT / "STEP42_6hour_solar_export_consistency_missingness.csv")


In [ ]:
# ============================================================
# CELL 10: 6-hour validation restricted to final 1,413 study households
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import pearsonr, spearmanr

OUT = Path("sixhour_outputs")

validation_rows = pd.read_csv(OUT / "sixhour_monthly_validation_rows.csv")
profile = pd.read_csv("ANALYSIS2_consumption_group_profile_dataset.csv")

validation_rows["household_ID"] = validation_rows["household_ID"].astype(str).str.strip()
profile["household_ID"] = profile["household_ID"].astype(str).str.strip()

final_households = set(profile["household_ID"])

val_final = validation_rows[
    validation_rows["household_ID"].isin(final_households)
].copy()

val_final = val_final.dropna(subset=["consumption", "sixh_import_kwh"]).copy()

val_final["abs_error_kwh"] = (
    val_final["sixh_import_kwh"] - val_final["consumption"]
).abs()

val_final["ape"] = np.where(
    val_final["consumption"] > 0,
    val_final["abs_error_kwh"] / val_final["consumption"],
    np.nan
)

if (
    len(val_final) >= 10
    and val_final["consumption"].nunique() > 1
    and val_final["sixh_import_kwh"].nunique() > 1
):
    pearson_r, pearson_p = pearsonr(
        val_final["consumption"],
        val_final["sixh_import_kwh"]
    )

    spearman_r, spearman_p = spearmanr(
        val_final["consumption"],
        val_final["sixh_import_kwh"]
    )
else:
    pearson_r = np.nan
    pearson_p = np.nan
    spearman_r = np.nan
    spearman_p = np.nan

final_validation_summary = pd.DataFrame([{
    "validation_scope": "final_1413_households_only",
    "matched_household_months": len(val_final),
    "matched_households": val_final["household_ID"].nunique(),
    "pearson_r": pearson_r,
    "pearson_p": pearson_p,
    "spearman_r": spearman_r,
    "spearman_p": spearman_p,
    "median_abs_error_kwh": val_final["abs_error_kwh"].median(),
    "median_ape": val_final["ape"].median(),
    "mean_ape": val_final["ape"].mean()
}])

display(final_validation_summary)

val_final.to_csv(
    OUT / "STEP42_6hour_validation_rows_final_1413.csv",
    index=False
)

final_validation_summary.to_csv(
    OUT / "STEP42_6hour_validation_summary_final_1413.csv",
    index=False
)

print("Saved final 1413 validation outputs.")

In [ ]:
# ============================================================
# CELL 11: Near-zero consumption audit for potentially inactive or anomalous accounts
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

OUT = Path("sixhour_outputs")

profile = pd.read_csv("ANALYSIS2_consumption_group_profile_dataset.csv")
profile["household_ID"] = profile["household_ID"].astype(str).str.strip()

profile["mean_monthly_kwh"] = pd.to_numeric(
    profile["mean_monthly_kwh"],
    errors="coerce"
)

near_zero_summary = pd.DataFrame([{
    "sample_households": profile["household_ID"].nunique(),
    "zero_mean_monthly_kwh": int((profile["mean_monthly_kwh"] == 0).sum()),
    "below_1_kwh_mean_monthly": int((profile["mean_monthly_kwh"] < 1).sum()),
    "below_5_kwh_mean_monthly": int((profile["mean_monthly_kwh"] < 5).sum()),
    "below_10_kwh_mean_monthly": int((profile["mean_monthly_kwh"] < 10).sum()),
    "median_mean_monthly_kwh": profile["mean_monthly_kwh"].median(),
    "min_mean_monthly_kwh": profile["mean_monthly_kwh"].min(),
    "max_mean_monthly_kwh": profile["mean_monthly_kwh"].max()
}])

display(near_zero_summary)

near_zero_households = profile[
    profile["mean_monthly_kwh"] < 1
].copy()

display_cols = [
    c for c in [
        "household_ID",
        "mean_monthly_kwh",
        "consumption_tertile",
        "consumption_group",
        "DEPOT",
        "BRANCH"
    ]
    if c in near_zero_households.columns
]

display(near_zero_households[display_cols].head(30))

near_zero_summary.to_csv(
    OUT / "STEP42_near_zero_account_audit_summary.csv",
    index=False
)

near_zero_households.to_csv(
    OUT / "STEP42_near_zero_account_audit_households.csv",
    index=False
)

print("Saved near-zero potentially inactive/anomalous account audit.")

In [ ]:
# ============================================================
# CELL 13: Descriptive panel time-series analysis
# Monthly 6-hour reconstructed import by consumption group
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests
from pathlib import Path

OUT = Path("sixhour_outputs")
FIG_DIR = Path("final_figures")
FIG_DIR.mkdir(exist_ok=True)

df = pd.read_csv(OUT / "STEP42_6hour_monthly_grouped.csv")

df["household_ID"] = df["household_ID"].astype(str).str.strip()
df["month"] = df["month"].astype(str)
df["month_dt"] = pd.to_datetime(df["month"] + "-01", errors="coerce")
df["consumption_group"] = df["consumption_group"].astype(str).str.strip()

df["sixh_import_kwh"] = pd.to_numeric(df["sixh_import_kwh"], errors="coerce")

group_order = ["Low", "Medium", "High"]

df = df[
    df["consumption_group"].isin(group_order)
    & df["month_dt"].notna()
    & df["sixh_import_kwh"].notna()
].copy()

# ------------------------------------------------------------
# 1. Monthly summary with confidence intervals
# ------------------------------------------------------------

monthly_ts = (
    df.groupby(["month", "month_dt", "consumption_group"], as_index=False)
    .agg(
        households=("household_ID", "nunique"),
        mean_import_kwh=("sixh_import_kwh", "mean"),
        median_import_kwh=("sixh_import_kwh", "median"),
        std_import_kwh=("sixh_import_kwh", "std")
    )
)

monthly_ts["se_import_kwh"] = monthly_ts["std_import_kwh"] / np.sqrt(monthly_ts["households"])

# Approximate 95% confidence interval for the group mean.
# Monthly kWh is right-skewed, so this is an asymptotic normal approximation,
# acceptable here because each monthly group has a large household count.
monthly_ts["ci95_low"] = monthly_ts["mean_import_kwh"] - 1.96 * monthly_ts["se_import_kwh"]
monthly_ts["ci95_high"] = monthly_ts["mean_import_kwh"] + 1.96 * monthly_ts["se_import_kwh"]

monthly_ts["consumption_group"] = pd.Categorical(
    monthly_ts["consumption_group"],
    categories=group_order,
    ordered=True
)

monthly_ts = monthly_ts.sort_values(["month_dt", "consumption_group"])

monthly_ts.to_csv(
    OUT / "STEP42_timeseries_monthly_import_by_group_with_ci.csv",
    index=False
)

print("Monthly time-series summary:")
display(monthly_ts.head(15))

# ------------------------------------------------------------
# 2. Check whether Low < Medium < High holds every month
# ------------------------------------------------------------

wide_mean = monthly_ts.pivot(
    index="month",
    columns="consumption_group",
    values="mean_import_kwh"
).reset_index()

wide_mean["low_lt_medium"] = wide_mean["Low"] < wide_mean["Medium"]
wide_mean["medium_lt_high"] = wide_mean["Medium"] < wide_mean["High"]
wide_mean["ordered_low_medium_high"] = (
    wide_mean["low_lt_medium"] & wide_mean["medium_lt_high"]
)

wide_mean["high_to_low_ratio"] = wide_mean["High"] / wide_mean["Low"]
wide_mean["high_to_medium_ratio"] = wide_mean["High"] / wide_mean["Medium"]

wide_mean.to_csv(
    OUT / "STEP42_timeseries_group_order_check.csv",
    index=False
)

print("Monthly group-order check:")
display(wide_mean)

print("All months preserve Low < Medium < High:",
      bool(wide_mean["ordered_low_medium_high"].all()))

# ------------------------------------------------------------
# 3. Month-wise Kruskal-Wallis tests across groups
# ------------------------------------------------------------

test_rows = []

for month, g in df.groupby("month"):
    arrays = []

    for group in group_order:
        vals = g.loc[g["consumption_group"] == group, "sixh_import_kwh"].dropna().values
        if len(vals) > 5:
            arrays.append(vals)

    if len(arrays) == 3:
        H, p = kruskal(*arrays)
    else:
        H, p = np.nan, np.nan

    test_rows.append({
        "month": month,
        "kruskal_H": H,
        "p_value": p,
        "n_household_months": len(g),
        "n_households": g["household_ID"].nunique()
    })

monthly_tests = pd.DataFrame(test_rows).sort_values("month")

valid_p = monthly_tests["p_value"].notna()

monthly_tests.loc[valid_p, "holm_p_value"] = multipletests(
    monthly_tests.loc[valid_p, "p_value"],
    method="holm"
)[1]

monthly_tests["significant_after_holm"] = monthly_tests["holm_p_value"] < 0.05

monthly_tests.to_csv(
    OUT / "STEP42_timeseries_monthwise_kruskal_tests.csv",
    index=False
)

print("Month-wise Kruskal-Wallis tests:")
display(monthly_tests)

# ------------------------------------------------------------
# 4. Plot mean monthly import with 95% CI
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(8.5, 5.0))

months_sorted = sorted(monthly_ts["month_dt"].unique())
x_labels = [pd.to_datetime(m).strftime("%Y-%m") for m in months_sorted]
x = np.arange(len(months_sorted))

for group in group_order:
    tmp = monthly_ts[monthly_ts["consumption_group"] == group].sort_values("month_dt")

    y = tmp["mean_import_kwh"].values
    y_low = tmp["ci95_low"].values
    y_high = tmp["ci95_high"].values

    ax.plot(
        x,
        y,
        marker="o",
        linewidth=1.8,
        label=group
    )

    ax.fill_between(
        x,
        y_low,
        y_high,
        alpha=0.12
    )

ax.set_xticks(x)
ax.set_xticklabels(x_labels, rotation=45)
ax.set_xlabel("Month")
ax.set_ylabel("Mean reconstructed 6-hour import (kWh)")
ax.set_title("Panel time-series of reconstructed monthly import by consumption group")
ax.legend(title="Consumption group", frameon=False)

plt.tight_layout()

fig_path = FIG_DIR / "Figure_34_timeseries_monthly_import_by_group_ci.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:")
print(OUT / "STEP42_timeseries_monthly_import_by_group_with_ci.csv")
print(OUT / "STEP42_timeseries_group_order_check.csv")
print(OUT / "STEP42_timeseries_monthwise_kruskal_tests.csv")
print(fig_path)

In [ ]:
# ============================================================
# CELL 14: 6-hour billing-validation accuracy by PUCSL band
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

OUT = Path("sixhour_outputs")

validation = pd.read_csv(
    OUT / "STEP42_6hour_validation_rows_final_1413.csv"
)

validation["consumption"] = pd.to_numeric(
    validation["consumption"],
    errors="coerce"
)

validation["sixh_import_kwh"] = pd.to_numeric(
    validation["sixh_import_kwh"],
    errors="coerce"
)

validation = validation.dropna(
    subset=[
        "household_ID",
        "month",
        "consumption",
        "sixh_import_kwh"
    ]
).copy()

band_order = [
    "0–30 kWh",
    "31–60 kWh",
    "61–90 kWh",
    "91–120 kWh",
    "121–180 kWh",
    ">180 kWh"
]

validation["pucsl_consumption_band"] = pd.cut(
    validation["consumption"],
    bins=[
        -np.inf,
        30,
        60,
        90,
        120,
        180,
        np.inf
    ],
    labels=band_order,
    include_lowest=True,
    right=True,
    ordered=True
)

validation["signed_error_kwh"] = (
    validation["sixh_import_kwh"]
    - validation["consumption"]
)

validation["absolute_error_kwh"] = (
    validation["signed_error_kwh"].abs()
)

validation["ape"] = np.where(
    validation["consumption"] > 0,
    validation["absolute_error_kwh"]
    / validation["consumption"],
    np.nan
)

validation_by_band = (
    validation
    .groupby(
        "pucsl_consumption_band",
        observed=False
    )
    .agg(
        matched_household_months=("household_ID", "count"),
        households=("household_ID", "nunique"),
        median_billing_kwh=("consumption", "median"),
        median_sixh_import_kwh=("sixh_import_kwh", "median"),
        median_absolute_error_kwh=("absolute_error_kwh", "median"),
        median_ape=("ape", "median"),
        mean_signed_error_kwh=("signed_error_kwh", "mean")
    )
    .reset_index()
)

validation_by_band.to_csv(
    OUT / "STEP42_6hour_validation_accuracy_by_pucsl_band.csv",
    index=False
)

print("6-hour validation accuracy by PUCSL band:")
display(validation_by_band)

In [ ]:
# ============================================================
# CELL 15: Corrected gap-sensitivity validation
# ============================================================
# This sensitivity analysis uses the corrected monthly allocation:
# - same-month valid intervals;
# - adjacent cross-month intervals <=24 h split by time;
# - long/non-adjacent cross-month intervals excluded.

import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import pearsonr, spearmanr

OUT = Path("sixhour_outputs")

allocated = pd.read_csv(
    OUT / "sixhour_month_allocated_intervals.csv",
    low_memory=False
)

allocated["household_ID"] = (
    allocated["household_ID"].astype(str).str.strip()
)

allocated["month"] = allocated["month"].astype(str)

allocated["gap_hours"] = pd.to_numeric(
    allocated["gap_hours"],
    errors="coerce"
)

allocated["total_import_delta_kwh"] = pd.to_numeric(
    allocated["total_import_delta_kwh"],
    errors="coerce"
)

profile = pd.read_csv(
    "ANALYSIS2_consumption_group_profile_dataset.csv"
)

profile["household_ID"] = (
    profile["household_ID"].astype(str).str.strip()
)

final_ids = set(profile["household_ID"])

monthly = pd.read_csv("monthly_consumption.csv")

monthly["household_ID"] = (
    monthly["household_ID"].astype(str).str.strip()
)

monthly["month"] = (
    pd.to_datetime(
        monthly["month"],
        errors="coerce"
    )
    .dt.to_period("M")
    .astype(str)
)

monthly["consumption"] = (
    monthly["consumption"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.strip()
)

monthly["consumption"] = pd.to_numeric(
    monthly["consumption"],
    errors="coerce"
)

monthly = monthly[
    monthly["household_ID"].isin(final_ids)
    & monthly["month"].between(
        "2023-04",
        "2024-03"
    )
].copy()

monthly = (
    monthly
    .dropna(
        subset=[
            "household_ID",
            "month",
            "consumption"
        ]
    )
    .groupby(
        ["household_ID", "month"],
        as_index=False
    )
    .agg(
        consumption=("consumption", "sum")
    )
)

allocated = allocated[
    allocated["household_ID"].isin(final_ids)
].copy()

main_allocated_energy = (
    allocated["total_import_delta_kwh"].sum()
)

def evaluate_allocated_scenario(
    scenario,
    maximum_gap_hours=None
):
    selected = allocated.copy()

    if maximum_gap_hours is not None:
        selected = selected[
            selected["gap_hours"]
            <= maximum_gap_hours
        ].copy()

    reconstructed = (
        selected
        .groupby(
            ["household_ID", "month"],
            as_index=False
        )
        .agg(
            sixh_import_kwh=(
                "total_import_delta_kwh",
                "sum"
            )
        )
    )

    validation = monthly.merge(
        reconstructed,
        on=["household_ID", "month"],
        how="inner",
        validate="one_to_one"
    )

    validation = validation.dropna(
        subset=[
            "consumption",
            "sixh_import_kwh"
        ]
    ).copy()

    validation["signed_error_kwh"] = (
        validation["sixh_import_kwh"]
        - validation["consumption"]
    )

    validation["absolute_error_kwh"] = (
        validation["signed_error_kwh"].abs()
    )

    validation["ape"] = np.where(
        validation["consumption"] > 0,
        validation["absolute_error_kwh"]
        / validation["consumption"],
        np.nan
    )

    pearson_r, pearson_p = pearsonr(
        validation["consumption"],
        validation["sixh_import_kwh"]
    )

    spearman_r, spearman_p = spearmanr(
        validation["consumption"],
        validation["sixh_import_kwh"]
    )

    retained_energy = (
        selected["total_import_delta_kwh"].sum()
    )

    return {
        "scenario": scenario,
        "maximum_gap_hours": (
            "No additional maximum"
            if maximum_gap_hours is None
            else maximum_gap_hours
        ),
        "retained_import_kwh": retained_energy,
        "retained_main_energy_pct": (
            100
            * retained_energy
            / main_allocated_energy
        ),
        "matched_household_months": len(validation),
        "matched_households": (
            validation["household_ID"].nunique()
        ),
        "pearson_r": pearson_r,
        "pearson_p": pearson_p,
        "spearman_r": spearman_r,
        "spearman_p": spearman_p,
        "median_absolute_error_kwh": (
            validation["absolute_error_kwh"].median()
        ),
        "median_ape": validation["ape"].median(),
        "mean_ape": validation["ape"].mean(),
        "mean_signed_error_kwh": (
            validation["signed_error_kwh"].mean()
        )
    }

scenarios = [
    ("Corrected main allocation", None),
    ("Corrected allocation, gaps <=48 h", 48),
    ("Corrected allocation, gaps <=24 h", 24)
]

corrected_gap_summary = pd.DataFrame([
    evaluate_allocated_scenario(
        scenario,
        maximum_gap
    )
    for scenario, maximum_gap in scenarios
])

corrected_gap_summary.to_csv(
    OUT
    / "STEP42_6hour_gap_sensitivity_summary_CORRECTED.csv",
    index=False
)

print("Corrected unequal-gap sensitivity:")
display(corrected_gap_summary)

allocation_audit = pd.read_csv(
    OUT / "STEP42_cross_month_allocation_audit.csv"
)

print("Cross-month allocation audit:")
display(allocation_audit)


In [ ]:
# ============================================================
# CELL 12: Final six-hour audit evidence summary
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

OUT = Path("sixhour_outputs")

delta_quality = pd.read_csv(OUT / "sixhour_delta_quality_summary.csv")
val_1413 = pd.read_csv(
    OUT / "STEP42_6hour_validation_summary_final_1413.csv"
)
solar_audit = pd.read_csv(
    OUT / "STEP42_6hour_solar_export_consistency_threshold_summary.csv"
)
near_zero = pd.read_csv(
    OUT / "STEP42_near_zero_account_audit_summary.csv"
)
tou_tests = pd.read_csv(OUT / "STEP42_6hour_tou_share_tests.csv")


def safe_lookup(
    df,
    filter_col,
    filter_value,
    value_col,
    default=np.nan,
):
    """Safely return one value from a summary table."""
    if df is None or df.empty:
        return default
    if filter_col not in df.columns or value_col not in df.columns:
        return default

    values = df.loc[df[filter_col] == filter_value, value_col]
    if values.empty:
        return default

    return values.iloc[0]


import_valid_rate = float(
    safe_lookup(
        delta_quality,
        "delta_col",
        "total_import_delta_kwh",
        "valid_rate",
        default=np.nan,
    )
)

validation_pearson = float(val_1413.loc[0, "pearson_r"])
validation_spearman = float(val_1413.loc[0, "spearman_r"])
validation_median_ape = float(val_1413.loc[0, "median_ape"])
validation_households = int(val_1413.loc[0, "matched_households"])

solar_export_gt1_pct = float(
    safe_lookup(
        solar_audit,
        "solar_export_group",
        "Solar + Export",
        "export_gt_1kwh_pct",
        default=np.nan,
    )
)
nonsolar_export_gt1_pct = float(
    safe_lookup(
        solar_audit,
        "solar_export_group",
        "Non-solar / Non-export",
        "export_gt_1kwh_pct",
        default=np.nan,
    )
)

near_zero_below1 = int(
    near_zero.loc[0, "below_1_kwh_mean_monthly"]
)

if "significant_after_holm" not in tou_tests.columns:
    raise ValueError(
        "STEP42_6hour_tou_share_tests.csv must contain "
        "significant_after_holm. Rerun the corrected ToU test cell first."
    )

tou_all_significant = bool(
    tou_tests["significant_after_holm"].fillna(False).all()
)
tou_max_holm_p = float(tou_tests["holm_p_value"].max())
tou_min_effect = float(tou_tests["epsilon_squared"].min())
tou_max_effect = float(tou_tests["epsilon_squared"].max())

decision_summary = pd.DataFrame([
    {
        "check": "Six-hour cumulative-register differencing",
        "result": (
            f"Total-import valid delta rate = {import_valid_rate:.3f}"
        ),
        "interpretation": (
            "Cumulative-register subtraction was retained after excluding "
            "negative deltas, non-positive gaps, missing deltas and rates "
            "above the documented anomaly cap."
        ),
    },
    {
        "check": "Six-hour versus monthly concordance, final 1,413 sample",
        "result": (
            f"Pearson r = {validation_pearson:.3f}; "
            f"Spearman r = {validation_spearman:.3f}; "
            f"median APE = {validation_median_ape:.3f}; "
            f"matched households = {validation_households}"
        ),
        "interpretation": (
            "The separate six-hour cumulative-register reconstruction showed "
            "strong concordance with monthly billing in the matched analytical sample."
        ),
    },
    {
        "check": "Day, Peak and Off-peak register shares",
        "result": (
            f"All Holm-adjusted comparisons significant = {tou_all_significant}; "
            f"maximum adjusted p = {tou_max_holm_p:.3g}; "
            f"epsilon-squared range = {tou_min_effect:.4f}–{tou_max_effect:.4f}"
        ),
        "interpretation": (
            "The billing-defined Low, Medium and High groups differed in broad "
            "register-share composition. Because the groups were defined using "
            "monthly consumption, this is supporting descriptive structure rather "
            "than independent discovery of new groups."
        ),
    },
    {
        "check": "Solar/export register consistency",
        "result": (
            f"Solar + Export with register export >1 kWh = "
            f"{solar_export_gt1_pct:.1f}%; "
            f"Non-solar / Non-export with register export >1 kWh = "
            f"{nonsolar_export_gt1_pct:.1f}%"
        ),
        "interpretation": (
            "Export activity was concentrated in the recorded Solar + Export "
            "group. Missing survey or metadata states were retained as unknown, "
            "and known-status mismatches were exported for audit."
        ),
    },
    {
        "check": "Near-zero account audit",
        "result": (
            f"Households below 1 kWh mean monthly consumption = "
            f"{near_zero_below1}"
        ),
        "interpretation": (
            "These households were flagged as potentially inactive or anomalous. "
            "Vacancy was not directly observed and is not inferred from consumption alone."
        ),
    },
    {
        "check": "Final evidence role",
        "result": "SUPPLEMENTARY SOURCE AUDIT AND BILLING-CONCORDANCE EVIDENCE",
        "interpretation": (
            "Use this notebook for the 1,413-household high-consumption profiling "
            "cohort. Do not present it as the main clustering model, a four-archetype "
            "validation notebook, a forecasting system or causal evidence."
        ),
    },
])

display(decision_summary)

decision_summary.to_csv(
    OUT / "STEP42_FINAL_6hour_audit_decision_summary.csv",
    index=False,
)

print("Saved:", OUT / "STEP42_FINAL_6hour_audit_decision_summary.csv")
